# Song Popularity Prediction

In this notebook, we will work with the processed file created and train some model on it.<br><br>

**1. Importing Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

**2. Loading the dataset**

In [2]:
df = pd.read_csv('../data/processed/processed_spotify_dataset.csv')  
df.sample(5)

,popularity,duration_ms,explicit,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,...,spanish,study,swedish,synth-pop,tango,techno,trance,trip-hop,turkish,world-music
76215,25,-0.511092,0,0.447117,10,1.124324,0,-0.424676,1.344851,1.315721,...,0,0,0,0,0,0,0,0,0,0
94139,51,0.028042,0,-0.028133,5,-0.075925,1,0.621561,-0.136665,-0.593862,...,0,0,0,0,0,0,0,0,0,0
111089,60,-0.685771,0,0.212408,9,0.632655,1,0.553117,0.219270,1.487459,...,0,0,0,0,0,0,0,1,0,0
3303,76,-0.014794,0,0.842055,2,0.498584,1,-0.521758,0.505116,-0.593075,...,0,0,0,0,0,0,0,0,0,0
16625,31,4.758342,0,-0.568947,4,-0.709186,1,0.968509,1.378694,-0.593862,...,0,0,0,0,0,0,0,0,0,0


**3. Splitting Train - Test dataset and Splitting target feature**

In [3]:
X = df.drop(columns=['popularity'])
y = df['popularity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**4. Defining function to predict and evaluate for each model**

In [4]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    return mae, rmse, r2


models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "Decision Tree": DecisionTreeRegressor(max_depth=5)
}

**5. Create a dataframe for storing results for each model**

In [5]:
results = []

for name, model in models.items():
    mae, rmse, r2 = evaluate_model(model, X_train, X_test, y_train, y_test)
    
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_R2_Mean": np.mean(cv_scores)
    })

Results so far

In [6]:
results_df = pd.DataFrame(results)
print(results_df)

               Model       MAE       RMSE        R2    CV_R2_Mean
0  Linear Regression  6.119011  11.557318  0.730751 -8.627871e+21
1              Ridge  6.118830  11.557322  0.730751  7.252225e-01
2              Lasso  5.986055  11.608505  0.728361  7.263323e-01
3      Decision Tree  6.241359  11.704539  0.723848  7.069676e-01


**1. Linear Regression**
* Interpretation:<br>
On training/test split → looks fine (~0.73 R²)
But cross-validation → completely unstable / broken
<br>
* Why this happens:<br>
Likely due to:<br>
    -- Multicollinearity<br>
    -- Outliers<br>
    -- Data leakage or improper scaling<br>
    -- Numerical instability<br>

* Conclusion:<br>
Do NOT trust this model despite decent R² — it's failing badly in generalization.<br><br>

**2. Ridge Regression**
* Interpretation:<br>
Very stable across folds<br>
Regularization fixed Linear Regression’s instability<br>

* Conclusion: <br>
Best overall choice — stable + consistent + interpretable<br>

**3. Lasso Regression**
* Interpretation:<br>
Slightly better MAE → better average predictions<br>
Slight drop in R² → explains slightly less variance<br>
Performs feature selection (some coefficients → 0)<br>

* Conclusion:<br>
Great if you want: <br>
    -- Simpler model<br>
    -- Feature selection<br>
    -- Slightly better prediction accuracy<br>
    
**4. Decision Tree**
* Interpretation:<br>
Performs worse than linear models<br>
Slight overfitting tendency (trees often do this)<br>

* Conclusion:<br>
Not ideal here — your data likely has linear relationships<br>



**6. Hyperparameter Tuning**

In [7]:
param_grids = {
    "Lasso": {
        "alpha": [0.001, 0.01, 0.1, 1, 10]
    },
    
    "Ridge": {
        "alpha": [0.001, 0.01, 0.1, 1, 10]
    },
    
    "Decision Tree": {
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }
}

In [8]:
results = []

for name, model in models.items():
    print(f"Tuning {name}...")

    if name in param_grids:
        grid = GridSearchCV(
            model,
            param_grids[name],
            cv=5,
            scoring='r2',
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
        best_params = grid.best_params_
    else:
        model.fit(X_train, y_train)
        best_model = model
        best_params = "Default"

    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    cv_score = cross_val_score(best_model, X, y, cv=5, scoring='r2').mean()

    results.append({
        "Model": name,
        "Best Params": best_params,
        "MAE": mae,
        "RMSE": rmse,
        "Test R2": r2,
        "CV R2 Mean": cv_score
    })

Tuning Linear Regression...
Tuning Ridge...
Tuning Lasso...
Tuning Decision Tree...


In [9]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="CV R2 Mean", ascending=False)

results_df.head(10)

,Model,Best Params,MAE,RMSE,Test R2,CV R2 Mean
2,Lasso,{'alpha': 0.001},6.100406,11.557462,0.730744,7.254661e-01
1,Ridge,{'alpha': 10},6.117112,11.557309,0.730751,7.253666e-01
3,Decision Tree,"{'max_depth': 5, 'min_samples_leaf': 4, 'min_s...",6.241359,11.704539,0.723848,7.070055e-01
0,Linear Regression,Default,6.119011,11.557318,0.730751,-1.192010e+20


**What changed after tuning?**
* Linear Regression <br>
Still the same (no hyperparameters)<br>
**This confirms:**<br>
The issue is not hyperparameters.
It’s something structural in your data (very likely multicollinearity or scaling issues)<br>
**Conclusion:** Still unreliable → ignore it<br><br>

**Best Models After Tuning**
* Lasso (alpha = 0.001)

**What this means:**<br>
    -- Best generalization<br>
    -- Slightly better prediction accuracy<br>
    -- Very small alpha → behaves close to Linear Regression but with just enough regularization<br>

**Why it won:**<br>
    -- Removed unnecessary noise features<br>
    -- Prevented instability seen in Linear Regression<br><br>

* Ridge (alpha = 10)

**What this means:**<br>
    --Almost identical to Lasso<br>
    --Slightly worse MAE but extremely stable<br>

**Why it works well:**<br>
    --Handles multicollinearity better than Linear Regression<br>
    --Keeps all features (no elimination)<br><br>

* Decision Tree (tuned)

**What tuning did:**<br>
    -- Reduced overfitting (good)
    -- But still underperforms linear models


**What should you do next?**<br>
1. Check feature importance<br>
**For Lasso:**<br>
    -- See which coefficients became 0<br>
    -- Those features are irrelevant → you can drop them<br>

2. Try ElasticNet (best of both worlds)<br>
Combines Ridge + Lasso<br>

3. Try stronger models <br>
    -- Random Forest<br>
    -- XGBoost / LightGBM <br>

4. Investigate Linear Regression failure <br>
    -- Check correlation matrix<br>
    -- Check feature scaling<br>
    -- Check outliers<br>

5. Use Polynomial feature


**7. Using Polynomial Features**

In [10]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

In [11]:
X_poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out())
X_poly_df.head(10)

,duration_ms,explicit,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,...,trance^2,trance trip-hop,trance turkish,trance world-music,trip-hop^2,trip-hop turkish,trip-hop world-music,turkish^2,turkish world-music,world-music^2
0,0.201339,0.0,-0.827804,1.0,0.132593,0.0,1.483006,-0.933744,-0.593842,1.261167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.920362,0.0,-1.675387,1.0,-1.658038,1.0,0.507178,1.492172,-0.593753,-0.701486,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.036445,0.0,-1.158407,0.0,-0.538749,1.0,-0.059980,-0.018783,-0.593862,-0.475311,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.149492,0.0,-1.904202,0.0,-1.803951,1.0,-0.793222,1.471943,-0.592475,-0.282743,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.189670,0.0,-0.889175,2.0,-0.528464,1.0,-0.162439,0.778256,-0.593862,-0.986411,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.005840,0.0,-0.758059,6.0,-0.351552,1.0,1.048117,0.274377,-0.593862,0.308396,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.186715,0.0,-1.719048,2.0,-0.354711,1.0,-0.828643,1.418121,-0.593805,-0.850103,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.339630,0.0,-0.885800,11.0,-0.459311,1.0,-0.565724,0.968792,-0.593862,-0.757102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,-0.313582,0.0,-0.985294,0.0,-0.328884,1.0,-0.766955,0.291191,-0.593862,-0.062877,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,-0.102432,0.0,-0.177002,1.0,0.126289,1.0,-1.109457,0.674587,-0.513124,-1.147901,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
results2 = []

for name, model in models.items():
    mae, rmse, r2 = evaluate_model(model, X_train, X_test, y_train, y_test)
    
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    
    results2.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_R2_Mean": np.mean(cv_scores)
    })

In [13]:
results_df2 = pd.DataFrame(results2)
results_df2.head(10)

,Model,MAE,RMSE,R2,CV_R2_Mean
0,Linear Regression,6.119011,11.557318,0.730751,-8.627871e+21
1,Ridge,6.118830,11.557322,0.730751,7.252225e-01
2,Lasso,5.986055,11.608505,0.728361,7.263323e-01
3,Decision Tree,6.241359,11.704539,0.723848,7.065483e-01


The result does not change much

**8. Applying ElasticNet**

In [14]:
from sklearn.linear_model import ElasticNet

model = ElasticNet(max_iter=10000)

In [15]:
param_grid = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

In [17]:
grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=ElasticNet(max_iter=10000), n_jobs=-1,
             param_grid={'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
                         'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
             scoring='r2')

In [18]:
print("Best Params:", grid.best_params_)
best_model = grid.best_estimator_

Best Params: {'alpha': 0.001, 'l1_ratio': 0.1}


In [19]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 6.1039263158392085
RMSE: 11.557758650550305
R2: 0.7307304585922308


This is also giving same result